# Comparison: trained h-network vs true cos(delta_D)

Loads a trained Symmetric SIREN h-network and compares its output to the
true $\cos\Delta\delta$ from the isobar amplitude model across the square
Dalitz plot.

In [ ]:
import numpy as np
import sys, os
import torch
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Setup paths
NOTEBOOK_DIR = os.path.abspath('')
H_DIR = os.path.join(NOTEBOOK_DIR, '..')        # models/h_network/
MODELS_DIR = os.path.join(H_DIR, '..')           # models/
ROOT = os.path.join(MODELS_DIR, '..')            # repo root
sys.path.insert(0, ROOT)

from Amplitude import SquareDalitzPlot2
from DKpp import DKpp
from models.h_network.models import SymmetricSIREN

In [ ]:
# Physics constants
M_D0, m_KS, m_pip, m_pim = 1.86484, 0.497611, 0.13957, 0.13957
sdp_obj = SquareDalitzPlot2(M_D0, m_KS, m_pip, m_pim)
IDX = (2, 3, 1)
S_TOTAL = M_D0**2 + m_KS**2 + 2 * m_pip**2

In [ ]:
# Load h-network (seed 1 has lowest RMSE)
SEED = 1
h_net = SymmetricSIREN(hidden=256, layers=5, omega_0=15.0)
h_net.load_state_dict(torch.load(
    os.path.join(H_DIR, f'weights/h_ensemble_sym_seed{SEED}.pth'),
    map_location='cpu', weights_only=False))
h_net.eval()
print(f'Loaded h-network seed {SEED}')
print(f'Parameters: {sum(p.numel() for p in h_net.parameters()):,}')

In [ ]:
# Build evaluation grid
N = 200
ms = np.linspace(0.01, 0.99, N)
ts = np.linspace(0.01, 0.99, N)
mm, tt = np.meshgrid(ms, ts)
grid = np.column_stack([mm.ravel(), tt.ravel()]).astype(np.float32)

# Convert to Dalitz coordinates
dp = np.empty((len(grid), 2), dtype=float)
for n, (mp, th) in enumerate(grid):
    dp[n, 0], dp[n, 1] = sdp_obj.M_from_MpT(mp, th, *IDX)
s23, s12 = dp[:, 0], dp[:, 1]
s13 = S_TOTAL - s23 - s12

# True cos(delta_D) from isobar model
dkpp = DKpp()
A12 = dkpp.full(np.column_stack([s12, s13]))
A13 = dkpp.full(np.column_stack([s13, s12]))
cos_dd_true = np.real(A12 * np.conj(A13)) / (np.abs(A12) * np.abs(A13) + 1e-30)

# h-network prediction
with torch.no_grad():
    h_pred = h_net(torch.from_numpy(grid)).numpy()

truth_2d = cos_dd_true.reshape(N, N)
pred_2d = h_pred.reshape(N, N)
resid_2d = pred_2d - truth_2d

rmse = np.sqrt(np.mean(resid_2d**2))
print(f'RMSE = {rmse:.4f}')

In [ ]:
# Publication figure
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],
    'text.latex.preamble': r'\usepackage{amsmath}',
    'axes.labelsize': 16,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.dpi': 150,
})

fig = plt.figure(figsize=(11, 9))
gs = GridSpec(2, 4, figure=fig, hspace=0.30, wspace=0.55)

ax0 = fig.add_subplot(gs[0, 0:2])
ax1 = fig.add_subplot(gs[0, 2:4])
ax2 = fig.add_subplot(gs[1, 1:3])

vmin, vmax = -1, 1

im0 = ax0.pcolormesh(ms, ts, truth_2d, cmap='RdBu_r', vmin=vmin, vmax=vmax,
                      shading='auto', rasterized=True)
ax0.set_title(r'$h_{\text{Amp.}}(m^\prime, \theta^\prime)$')
ax0.set_xlabel(r"$m'$")
ax0.set_ylabel(r"$\theta'$")
plt.colorbar(im0, ax=ax0, fraction=0.046, pad=0.04)

im1 = ax1.pcolormesh(ms, ts, pred_2d, cmap='RdBu_r', vmin=vmin, vmax=vmax,
                      shading='auto', rasterized=True)
ax1.set_title(r"$h_{\text{MLP}}(m^\prime, \theta^\prime)$ (SIREN)")
ax1.set_xlabel(r"$m'$")
ax1.set_yticklabels([])
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

rlim = 0.5
im2 = ax2.pcolormesh(ms, ts, resid_2d, cmap='RdBu_r', vmin=-rlim, vmax=rlim,
                      shading='auto', rasterized=True)
ax2.set_title(r'$h_{\text{MLP}} - h_{\text{Amp.}}$ (residual)')
ax2.set_xlabel(r"$m'$")
ax2.set_ylabel(r"$\theta'$")
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

plt.savefig(os.path.join(H_DIR, 'figures', 'h_vs_truth_pub.pdf'),
            bbox_inches='tight')
plt.show()
print('Saved to h_network/figures/h_vs_truth_pub.pdf')